# Visualizing the PE Mass-Ratio (q) Bias and Its Impact on HyperPE $\sigma_m$

## The Problem Chain

1. **PE prior mismatch**: PE uses flat $q \sim U(0.5, 1.0)$, but the true BNS population has $q$ strongly peaked near 1 (equal-mass systems)
2. **PE posterior biased low**: The flat PE prior over-explores low $q$, pulling the posterior median systematically below truth
3. **Source mass distribution broadened**: The broadened PE $q$ posteriors make the apparent mass distribution wider than reality
4. **$\sigma_m$ inflated in HyperPE**: The hierarchical step absorbs this extra width as larger $\sigma_m$ (0.12–0.13 instead of 0.09)

In [ ]:
import json
import glob
import math
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.interpolate import interp1d

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
})

RUNDIR = '/scratch/gpfs/ANDREASB/fy6204/GW/Workspace'
print("Imports OK")

In [ ]:
# --- Load injection truths ---
with open(f'{RUNDIR}/outdir_population_SEOBNR/accepted.jsonl') as f:
    events = [json.loads(line) for line in f]

n_events = len(events)
q_true = np.array([
    min(e['mass_2_source'], e['mass_1_source']) / max(e['mass_2_source'], e['mass_1_source'])
    for e in events
])
m1_src_true = np.array([e['mass_1_source'] for e in events])
m2_src_true = np.array([e['mass_2_source'] for e in events])

print(f"Loaded {n_events} events")
print(f"q_true:  median={np.median(q_true):.4f}, 90% CI=[{np.percentile(q_true,5):.4f}, {np.percentile(q_true,95):.4f}]")
print(f"m1_src:  median={np.median(m1_src_true):.4f}, std={np.std(m1_src_true):.4f}")
print(f"m2_src:  median={np.median(m2_src_true):.4f}, std={np.std(m2_src_true):.4f}")

In [ ]:
# --- Load PE posterior q and mass values ---
# Use the pre-computed q_bias_event_summary for efficiency
q_summary = pd.read_csv(f'{RUNDIR}/q_bias_event_summary_100.csv')

# Also load a few full PE posterior files for detailed distribution plots
pe_files = sorted(glob.glob(f'{RUNDIR}/outdir_population_run_SEOBNR/*_reweighted_posterior_augmented.csv'))

# Load all PE q samples for a subset of representative events
pe_q_samples = {}  # event_index -> q array
pe_m1s_samples = {}
pe_m2s_samples = {}

for i, f in enumerate(pe_files[:50]):  # first 50 events for speed
    df = pd.read_csv(f)
    # q from mass_ratio column
    pe_q_samples[i] = df['mass_ratio'].values.astype(float)
    pe_m1s_samples[i] = df['mass_1_source'].values.astype(float)
    pe_m2s_samples[i] = df['mass_2_source'].values.astype(float)

print(f"Loaded PE posteriors for {len(pe_q_samples)} events")
print(f"Typical samples per event: {len(pe_q_samples[0])}")
print(f"q_summary columns: {list(q_summary.columns)}")

## Figure 1: The PE Prior Mismatch

The PE uses $\pi_{\rm PE}(q) \propto 1$ (uniform in $q \in [0.5, 1.0]$).  
The true population draws component masses from a truncated Gaussian $\mathcal{N}_{[1.1,2.25]}(1.33, 0.09)$, which **implicitly defines a $q$ prior strongly peaked near 1**.

This mismatch means the PE over-explores low-$q$ configurations that the population model considers extremely unlikely.

In [ ]:
# --- Figure 1: PE prior vs population-implied q prior ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left panel: the two priors
ax = axes[0]

# PE prior: Uniform in q
q_grid = np.linspace(0.5, 1.0, 500)
pe_prior_q = np.ones_like(q_grid)  # uniform

# Population-implied q prior: Monte Carlo from component-mass truncated Gaussian
mu_inj, sig_inj = 1.33, 0.09
n_mc = 2_000_000
m1_mc = stats.truncnorm.rvs((1.1-mu_inj)/sig_inj, (2.25-mu_inj)/sig_inj, 
                             loc=mu_inj, scale=sig_inj, size=n_mc)
m2_mc = stats.truncnorm.rvs((1.1-mu_inj)/sig_inj, (2.25-mu_inj)/sig_inj,
                             loc=mu_inj, scale=sig_inj, size=n_mc)
q_pop = np.minimum(m1_mc, m2_mc) / np.maximum(m1_mc, m2_mc)

# KDE of population q
kde = stats.gaussian_kde(q_pop)
pop_prior_q = kde(q_grid)

ax.plot(q_grid, pe_prior_q / pe_prior_q.max(), 'C0-', lw=2.5, label='PE prior: $U(0.5, 1.0)$')
ax.plot(q_grid, pop_prior_q / pop_prior_q.max(), 'C3-', lw=2.5, 
        label=f'Population prior: $\mathcal{{N}}_{{[1.1,2.25]}}(1.33, 0.09)$')

# Mark the 90% interval of population q
q_lo, q_hi = np.percentile(q_pop, [5, 95])
ax.axvspan(q_lo, q_hi, alpha=0.12, color='C3', label=f'Pop 90% CI: [{q_lo:.3f}, {q_hi:.3f}]')
ax.axvline(0.5, color='gray', ls='--', alpha=0.5)

ax.set_xlabel('Mass ratio $q = m_2/m_1$')
ax.set_ylabel('Normalized prior density')
ax.set_title('PE prior vs Population-implied q prior')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0.48, 1.02)

# Right panel: q distribution of accepted events vs PE prior support
ax = axes[1]
ax.hist(q_true, bins=25, density=True, alpha=0.6, color='C3', 
        label=f'Accepted events (n={n_events})')
ax.axhline(1.0, color='C0', lw=2.5, ls='--', label='PE prior (uniform)')
ax.axvspan(q_lo, q_hi, alpha=0.08, color='C3')
ax.set_xlabel('Mass ratio $q$')
ax.set_ylabel('Density')
ax.set_title('Accepted event q distribution vs PE prior')
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_q1_prior_mismatch.png', dpi=150)
plt.show()
print(f"Population q 90% CI: [{q_lo:.4f}, {q_hi:.4f}]")
print(f"PE prior 90% CI: [0.54, 0.96]  (much wider at low q)")

## Figure 2: Per-Event q Bias

For each event, we compare $q_{\rm true}$ (from the injection) with the PE posterior median $q_{\rm PE}$.  
Events are colored by $\Delta q = q_{\rm PE} - q_{\rm true}$.

**Key observation**: nearly all high-$q$ events ($q_{\rm true} > 0.94$) have $q_{\rm PE} < q_{\rm true}$.
The systematic low bias is clearest where the prior mismatch is most severe (near $q=1$).

In [ ]:
# --- Figure 2: Per-event q_true vs q_PE scatter ---
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(2, 2, figure=fig, height_ratios=[3, 1], width_ratios=[3, 1])

# Main scatter
ax_main = fig.add_subplot(gs[0, 0])
dq = q_summary['q_pe_med'].values - q_summary['q_true'].values
q_pe_05 = q_summary['q_pe_05'].values
q_pe_95 = q_summary['q_pe_95'].values

# Color by dq
sc = ax_main.scatter(q_summary['q_true'], q_summary['q_pe_med'], 
                     c=dq, cmap='RdBu_r', vmin=-0.3, vmax=0.1,
                     s=30, edgecolors='k', linewidths=0.3, zorder=3)

# Error bars for 90% CI
for i in range(len(q_summary)):
    ax_main.plot([q_summary['q_true'].iloc[i], q_summary['q_true'].iloc[i]],
                 [q_pe_05[i], q_pe_95[i]], 
                 color='gray', alpha=0.25, lw=0.8, zorder=1)

# 1:1 line
ax_main.plot([0.78, 1.01], [0.78, 1.01], 'k--', lw=1, alpha=0.5, zorder=2)
# Mark q_true > 0.94 region
ax_main.axvspan(0.94, 1.0, alpha=0.06, color='C3')
ax_main.text(0.97, 0.725, 'high-q region\n(all biased low)', fontsize=9, 
             ha='center', color='C3', alpha=0.8)

ax_main.set_xlabel('True mass ratio $q_{\\rm true}$')
ax_main.set_ylabel('PE posterior median $q_{\\rm PE}$')
ax_main.set_title(f'Per-event q bias ({n_events} events)')
cbar = plt.colorbar(sc, ax=ax_main, label=r'$\Delta q = q_{\rm PE} - q_{\rm true}$')
ax_main.set_xlim(0.78, 1.01)
ax_main.set_ylim(0.70, 1.01)

# Top histogram: dq distribution
ax_top = fig.add_subplot(gs[0, 1])
ax_top.hist(dq, bins=30, orientation='horizontal', density=True, 
            color='C1', alpha=0.6, edgecolor='k', linewidth=0.5)
ax_top.axhline(0, color='k', ls='--', lw=1)
ax_top.set_xlabel('Density')
ax_top.set_ylabel(r'$\Delta q$')
ax_top.set_title(f'Median $\Delta q$ = {np.median(dq):.4f}')

# Bottom histogram: q_true distribution
ax_bottom = fig.add_subplot(gs[1, 0])
ax_bottom.hist(q_summary['q_true'], bins=30, density=True, 
               color='C0', alpha=0.6, edgecolor='k', linewidth=0.5)
ax_bottom.set_xlabel('$q_{\\rm true}$')
ax_bottom.set_ylabel('Density')

fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_q2_per_event_bias.png', dpi=150)
plt.show()

# --- Statistics ---
n_biased = np.sum(dq < -0.01)
n_highq = np.sum(q_summary['q_true'] > 0.94)
n_highq_biased = np.sum((q_summary['q_true'] > 0.94) & (dq < -0.01))
print(f"Events with q_PE < q_true by >0.01: {n_biased}/{n_events} ({100*n_biased/n_events:.0f}%)")
print(f"High-q events (q_true > 0.94): {n_highq}, ALL biased low: {n_highq_biased}/{n_highq}")
print(f"Median dq for high-q events: {np.median(dq[q_summary['q_true'] > 0.94]):.4f}")
print(f"Median dq for all events: {np.median(dq):.4f}")

## Figure 3: Source Mass Distribution Distortion

The q bias at the PE level propagates to the source-frame masses. When the PE gets $q$ wrong,
the component masses $m_1, m_2$ shift, broadening the apparent mass distribution.

Here we compare:
- **Truth**: the true source masses from `accepted.jsonl`
- **PE**: the PE posterior-median source masses for each event
- A narrow true $\sigma_m=0.09$ distribution vs a broader apparent one

In [ ]:
# --- Figure 3: Source mass distribution: truth vs PE ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: m1 source mass
ax = axes[0]
m1_all_true = np.concatenate([m1_src_true, m2_src_true])  # all component masses
m1_pe_med = q_summary['m1src_med'].values
m2_pe_med = q_summary['m2src_med'].values
m_all_pe = np.concatenate([m1_pe_med, m2_pe_med])

bins = np.linspace(1.1, 1.6, 30)
ax.hist(m1_all_true, bins=bins, density=True, alpha=0.5, color='C0', 
        label=f'Truth: std={np.std(m1_all_true):.4f}')
ax.hist(m_all_pe, bins=bins, density=True, alpha=0.5, color='C3',
        label=f'PE median: std={np.std(m_all_pe):.4f}')

# Overlay theoretical PDF
m_grid = np.linspace(1.1, 1.6, 200)
pdf_theory = stats.norm.pdf(m_grid, loc=1.33, scale=0.09)
ax.plot(m_grid, pdf_theory, 'k-', lw=2, label='Population: N(1.33, 0.09)')
ax.plot(m_grid, stats.norm.pdf(m_grid, loc=np.mean(m_all_pe), scale=np.std(m_all_pe)),
        'C3--', lw=2, alpha=0.7, label=f'PE fit: sigma={np.std(m_all_pe):.3f}')

ax.set_xlabel('Source-frame component mass')
ax.set_ylabel('Density')
ax.set_title('All component masses: Truth vs PE')
ax.legend(fontsize=8)

# Panel B: Per-event mass shift
ax = axes[1]
for i in range(min(100, len(m1_src_true))):
    ax.plot([m1_src_true[i], m1_pe_med[i]], 
            [m2_src_true[i], m2_pe_med[i]], 
            '-', color='gray', alpha=0.25, lw=0.5)
ax.scatter(m1_src_true, m2_src_true, c='C0', s=25, zorder=5, label='Truth', edgecolors='k', lw=0.3)
ax.scatter(m1_pe_med, m2_pe_med, c='C3', s=25, zorder=5, label='PE median', edgecolors='k', lw=0.3)
ax.plot([1.1, 1.55], [1.1, 1.55], 'k--', lw=0.8, alpha=0.4)
ax.set_xlabel('m1 source')
ax.set_ylabel('m2 source')
ax.set_title('Per-event mass shift (truth -> PE)')
ax.legend(fontsize=8)

# Panel C: Width comparison
ax = axes[2]
event_sigmas_true = []
event_sigmas_pe = []
for i in range(len(q_summary)):
    eidx = int(q_summary['index'].iloc[i]) - 1
    if eidx in pe_m1s_samples:
        m1s = pe_m1s_samples[eidx]
        m2s = pe_m2s_samples[eidx]
        all_m_pe_samples = np.concatenate([m1s, m2s])
        event_sigmas_pe.append(np.std(all_m_pe_samples))

ax.hist(event_sigmas_pe, bins=20, density=True, alpha=0.6, color='C3', edgecolor='k')
ax.axvline(0.09, color='C0', lw=2.5, ls='--', label='True sigma_m = 0.09')
ax.axvline(np.median(event_sigmas_pe), color='C3', lw=2.5, 
           label=f'Median PE mass std = {np.median(event_sigmas_pe):.4f}')
ax.set_xlabel('Per-event mass standard deviation')
ax.set_ylabel('Density')
ax.set_title('Per-event PE mass scatter')
ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_q3_mass_distortion.png', dpi=150)
plt.show()

print(f"Truth mass std: {np.std(m1_all_true):.4f}")
print(f"PE median mass std: {np.std(m_all_pe):.4f}")
print(f"Inflation ratio: {np.std(m_all_pe)/np.std(m1_all_true):.2f}x")
print(f"Per-event median PE mass std: {np.median(event_sigmas_pe):.4f}")

## Figure 4: The $\sigma_m$ Propagation Chain

The broadened apparent mass distribution feeds into HyperPE, which interprets the extra width
as larger population $\sigma_m$. Both EOSFIT and spectral pipelines show the same pattern:
truth-delta recovers $\sigma_m \approx 0.09$, but real PE infers $\sigma_m \approx 0.12-0.13$.

**The $\Delta q$ → $\sigma_m$ connection**: events with large q bias contribute more to
the apparent mass broadening, which HyperPE absorbs as inflated $\sigma_m$.

In [ ]:
# --- Figure 4: The sigma_m propagation ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: sigma_m from HyperPE results
ax = axes[0]

# These numbers from the actual runs
n_events_arr = np.array([10, 50, 100])
sigma_m_eosfit_truth = np.array([0.078, 0.089, 0.087])
sigma_m_eosfit_real = np.array([0.080, 0.124, 0.126])
sigma_m_spectral_truth = np.array([0.079, 0.093, 0.091])
sigma_m_spectral_real = np.array([0.08, 0.12, 0.13])

ax.plot(n_events_arr, sigma_m_eosfit_truth, 'o-', color='C0', lw=2, ms=8, 
        label='EOSFIT truth-delta')
ax.plot(n_events_arr, sigma_m_eosfit_real, 's--', color='C0', lw=2, ms=8, 
        label='EOSFIT real PE')
ax.plot(n_events_arr, sigma_m_spectral_truth, 'o-', color='C2', lw=2, ms=8,
        label='Spectral truth-delta')
ax.plot(n_events_arr, sigma_m_spectral_real, 's--', color='C2', lw=2, ms=8,
        label='Spectral real PE')

ax.axhline(0.09, color='k', ls=':', lw=1.5, label='Truth $\sigma_m = 0.09$')
ax.fill_between([8, 105], 0.09*0.85, 0.09*1.15, alpha=0.08, color='green', 
                label='$\pm 15\%$ band')

ax.set_xlabel('Number of events')
ax.set_ylabel('Inferred $\sigma_m$')
ax.set_title('HyperPE $\sigma_m$: truth-delta vs real PE')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(5, 105)
ax.set_ylim(0.06, 0.16)

# Right: per-event dq magnitude vs impact on mass width
ax = axes[1]
# For each event, compute how much its PE mass distribution differs from truth
mass_shift = np.sqrt((q_summary['m1src_med'].values - m1_src_true[:len(q_summary)])**2 +
                      (q_summary['m2src_med'].values - m2_src_true[:len(q_summary)])**2)

sc = ax.scatter(np.abs(dq), mass_shift, 
                c=q_summary['q_true'].values, cmap='viridis_r',
                s=40, edgecolors='k', linewidths=0.3)

# Color the high-q events specially
mask_highq = q_summary['q_true'] > 0.94
ax.scatter(np.abs(dq[mask_highq]), mass_shift[mask_highq],
           facecolors='none', edgecolors='C3', s=60, linewidths=1.5,
           label=f'High-q ($q_{{\\rm true}}>0.94$, n={mask_highq.sum()})')

ax.set_xlabel(r'$|\Delta q| = |q_{\rm PE} - q_{\rm true}|$')
ax.set_ylabel('Mass shift $||(m_1,m_2)_{\\rm PE} - (m_1,m_2)_{\\rm true}||$')
ax.set_title('q bias → mass shift per event')
cbar = plt.colorbar(sc, ax=ax, label='$q_{\\rm true}$')
ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_q4_sigma_m_impact.png', dpi=150)
plt.show()

print(f"High-q events: mean |dq| = {np.mean(np.abs(dq[mask_highq])):.4f}")
print(f"Low-q events:  mean |dq| = {np.mean(np.abs(dq[~mask_highq])):.4f}")
print(f"Mean mass shift for high-q: {np.mean(mass_shift[mask_highq]):.4f}")
print(f"Mean mass shift for low-q:  {np.mean(mass_shift[~mask_highq]):.4f}")

## Updated Summary: Two Independent PE Biases

### Problem 1: q Underestimated → σ_m Inflated

1. **PE prior mismatch** (Fig 1): PE uses flat $q \sim U(0.5, 1.0)$, but the BNS population has $q$ strongly peaked near 1
2. **Systematic q bias** (Fig 2): 83% events have $q_{\rm PE} < q_{\rm true}$. All 46 high-$q$ events biased low
3. **Mass distribution broadened** (Fig 3): Source mass $\sigma$ goes from 0.089 → 0.136 (1.5× wider)
4. **$\sigma_m$ inflated** (Fig 4): Both pipelines give $\sigma_m \approx 0.12-0.13$ for real PE (truth 0.09)

→ **Fixable by recycling** (PE mass prior is constant in det-frame masses, population model provides correct distribution). Practical issue: high-q posteriors are broad, importance sampling variance may be high. Solutions: more PE samples, or population-matched PE prior.

### Problem 2: dL Overestimated → H0 Underestimated

1. **dL systematically over-estimated** (Fig 5): 67/100 events, mean ratio 1.16
2. **~84% from likelihood, not prior** (Fig 6 left): 1/dL² prior correction only fixes ~16%
3. **dL-H0 anti-correlation** (Fig 6 right): $r=-0.62$, most events in dL-over/H0-under quadrant
4. **H0 shifted** (Fig 4, right panel): Both pipelines give $H_0 \approx 59-60$ for real PE (truth 67.66)

→ **Mostly NOT fixable by recycling**. The PE likelihood itself prefers lower redshift (dL×H0 systematically low). Solutions: investigate waveform systematics, or accept as known PE systematic.

### Why 10ev ≠ 50/100ev (Fig 7)

The 10-event result ($\mu_H \approx 76.5$) is **prior-dominated** (prior median 80, CI width 28.4). All event subsets have $H_0$ PE medians ~50-55 — the bias is always present. At 50+ events, enough data accumulates to overcome the prior and reveal the systematic bias.

### The two problems are INDEPENDENT

$\mathrm{corr}(\Delta q, d_L^{\rm PE}/d_L^{\rm true}) = +0.03$ (event-level) and $\mathrm{corr}(\sigma_m, \mu_H) = +0.006$ (HyperPE posterior). They happen to both bias HyperPE results, but through separate pathways.

## Problem 2: dL Overestimation → H0 Underestimation

This is a **separate, independent** bias from the q problem.  
While recycling correctly cancels the PE dL² prior, most of the dL bias (~84%) comes from the PE **likelihood** itself.

### The chain:
1. PE likelihood systematically prefers higher $d_L$ (mean ratio 1.16)
2. $d_L$ and $H_0$ are anti-correlated in the PE posterior ($r=-0.62$ across events)
3. $d_L \times H_0$ (which determines redshift) is biased low for 85/100 events
4. HyperPE absorbs the systematically low $H_0$ distribution → $\mu_H \approx 59-60$
5. Truth-delta recovers correct $H_0$, proving the HyperPE model is correct

### Key evidence that this is independent from the q problem:
- $\mathrm{corr}(\Delta q, d_L^{\mathrm{PE}}/d_L^{\mathrm{true}}) = +0.03$ (no correlation at event level)
- $\mathrm{corr}(\sigma_m, \mu_H) = +0.006$ (no correlation in HyperPE posterior)

In [ ]:
# --- Load dL data ---
dL_true_arr = np.array([e['luminosity_distance_mpc'] for e in events])
z_true_arr = np.array([e['z'] for e in events])

dL_ratio_all = q_summary['dL_ratio_med'].values
H0_med_all = q_summary['H0_sample_med'].values

# Load full PE posteriors for per-event dL analysis
pe_dL_medians = []
pe_dL_after_prior = []   # after 1/dL^2 reweight
pe_H0_medians = []
pe_dLscaled = []         # dL * H0 / 70
for f in pe_files:
    df = pd.read_csv(f)
    dL_s = df['luminosity_distance'].values.astype(float)
    H0_s = df['H0_sample'].values.astype(float)
    pe_dL_medians.append(np.median(dL_s))
    pe_H0_medians.append(np.median(H0_s))
    w = 1.0 / dL_s**2
    pe_dL_after_prior.append(np.average(dL_s, weights=w))
    pe_dLscaled.append(np.median(dL_s * H0_s / 70.0))

pe_dL_medians = np.array(pe_dL_medians)
pe_H0_medians = np.array(pe_H0_medians)
pe_dL_after_prior = np.array(pe_dL_after_prior)
pe_dLscaled = np.array(pe_dLscaled)

# Bias decomposition
dL_bias_total = pe_dL_medians[:len(dL_true_arr)] - dL_true_arr
dL_bias_prior_fixed = pe_dL_after_prior[:len(dL_true_arr)] - dL_true_arr
frac_fixed = np.median((dL_bias_total - dL_bias_prior_fixed)[np.abs(dL_bias_total) > 10] / 
                        dL_bias_total[np.abs(dL_bias_total) > 10])

# Simple correlation function
def corr(x, y):
    mx, my = np.mean(x), np.mean(y)
    return np.sum((x-mx)*(y-my)) / math.sqrt(np.sum((x-mx)**2)*np.sum((y-my)**2))

print(f"dL ratio: mean={np.mean(dL_ratio_all):.4f}, {np.sum(dL_ratio_all>1)}/100 events over-estimated")
print(f"dL-H0 corr: {corr(dL_ratio_all, H0_med_all):.4f}")
print(f"Prior-fixable: ~{frac_fixed*100:.0f}%")
print(f"dL*H0/70 PE/true ratio: {np.mean(pe_dLscaled/(dL_true_arr*67.66/70.0)[:len(pe_dLscaled)]):.4f}")

In [ ]:
# --- Figure 5: dL bias overview (ratio histogram + dL_true vs dL_PE) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: dL ratio histogram
ax = axes[0]
ax.hist(dL_ratio_all, bins=30, density=True, alpha=0.6, color='C1', edgecolor='k', linewidth=0.5)
ax.axvline(1.0, color='k', lw=2.5, ls='--', label='Truth (ratio=1)')
ax.axvline(np.median(dL_ratio_all), color='C1', lw=2.5, ls='-', label=f'Median={np.median(dL_ratio_all):.3f}')
ax.axvline(np.mean(dL_ratio_all), color='C1', lw=2, ls=':', label=f'Mean={np.mean(dL_ratio_all):.3f}')
n_high_dl = np.sum(dL_ratio_all > 1.0)
ax.set_xlabel('$d_L^{\\rm PE} / d_L^{\\rm true}$')
ax.set_ylabel('Density')
ax.set_title(f'dL ratio: {n_high_dl}/100 events over-estimated', fontweight='bold')
ax.legend(fontsize=9)

# Right: dL_true vs dL_PE scatter (all 100 events)
ax = axes[1]
dL_pct = (pe_dL_medians[:len(dL_true_arr)] - dL_true_arr) / dL_true_arr * 100
sc = ax.scatter(dL_true_arr, pe_dL_medians[:len(dL_true_arr)],
                c=dL_pct, cmap='RdYlBu_r', vmin=-20, vmax=60,
                s=30, edgecolors='k', linewidths=0.2, zorder=3)
dL_m = max(np.max(dL_true_arr), np.max(pe_dL_medians)) * 1.08
ax.plot([0, dL_m], [0, dL_m], 'k--', lw=1.5, alpha=0.5)
ax.fill_between([0, dL_m], [0, dL_m], [0, dL_m*1.3], alpha=0.04, color='C1')
ax.fill_between([0, dL_m], [0, dL_m*0.7], [0, dL_m], alpha=0.03, color='C0')
ax.text(dL_m*0.7, dL_m*1.18, 'Over-estimated', fontsize=9, color='C1', ha='center')
ax.text(dL_m*0.7, dL_m*0.78, 'Under-estimated', fontsize=9, color='C0', ha='center')
ax.set_xlabel('$d_L^{\\rm true}$ [Mpc]')
ax.set_ylabel('$d_L^{\\rm PE}$ median [Mpc]')
ax.set_title(f'All {n_events} events: dL true vs PE', fontweight='bold')
cbar = plt.colorbar(sc, ax=ax, label='dL bias [%]')

fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_dl1_overview.png', dpi=150)
plt.show()

In [ ]:
# --- Figure 6: Bias decomposition + dL-H0 anti-correlation ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Prior vs likelihood bias decomposition
ax = axes[0]
bins_dl = np.linspace(-200, 800, 40)
ax.hist(dL_bias_total, bins=bins_dl, density=True, alpha=0.5, color='C1',
        label=f'Total (median={np.median(dL_bias_total):.0f} Mpc)')
ax.hist(dL_bias_prior_fixed, bins=bins_dl, density=True, alpha=0.5, color='C3',
        label=f'After 1/dL$^2$ (median={np.median(dL_bias_prior_fixed):.0f} Mpc)')
ax.axvline(0, color='k', lw=2, ls='--')
ax.set_xlabel('$d_L^{\\rm PE} - d_L^{\\rm true}$ [Mpc]')
ax.set_ylabel('Density')
ax.set_title(f'Bias decomposition: ~{frac_fixed*100:.0f}% prior, ~{(1-frac_fixed)*100:.0f}% likelihood', fontweight='bold')
ax.legend(fontsize=8)

# Right: dL-H0 anti-correlation across all 100 events
ax = axes[1]
dL_fit_arr = np.linspace(0.6, 1.8, 50)
ax.plot(dL_fit_arr, 67.66 / dL_fit_arr, 'k--', lw=2, alpha=0.6,
        label='$H_0 = 67.66 / (d_L/d_L^{\\rm true})$')
ax.scatter(dL_ratio_all, H0_med_all, c='C1', alpha=0.5, s=25, edgecolors='k', lw=0.2, zorder=3)
ax.axhline(67.66, color='C0', ls=':', lw=1.5, alpha=0.7)
ax.axvline(1.0, color='C0', ls=':', lw=1.5, alpha=0.7)
ax.fill_between([0.6, 1.0], 50, 110, alpha=0.02, color='C0')
ax.fill_between([1.0, 1.8], 25, 70, alpha=0.05, color='C1')
ax.text(0.78, 100, 'dL under\n→ H$_0$ over', fontsize=8, color='C0', ha='center')
ax.text(1.45, 48, 'dL over\n→ H$_0$ under\n(most events)', fontsize=8, color='C1', ha='center')
ax.set_xlabel('$d_L^{\\rm PE} / d_L^{\\rm true}$')
ax.set_ylabel('$H_0$ PE posterior median')
ax.set_title(f'dL-H$_0$ anti-correlation (r={corr(dL_ratio_all, H0_med_all):.3f})', fontweight='bold')
ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_dl2_decomposition.png', dpi=150)
plt.show()

print(f"dL-H0 corr = {corr(dL_ratio_all, H0_med_all):.4f}")
print(f"dL×H0/70 ratio (PE/truth) = {np.mean(pe_dLscaled/(dL_true_arr*67.66/70.0)[:len(pe_dLscaled)]):.4f}")

## Figure 7: Why 10 Events Looks Different — Prior-Dominated, Not Correct

At first glance the 10-event result ($\mu_H \approx 76.5$) appears closer to truth (67.66) than 50/100 events ($\mu_H \approx 60$). But this is misleading:

- All event subsets have $H_0$ PE medians biased low (mean ~50-55)
- The 10-event HyperPE posterior has 90% CI width = 28.4 — far too broad to resolve the bias
- The $\mu_H$ prior is $U(10, 150)$ with median 80 — the 10-event posterior (76.5) is prior-dominated
- At 50+ events, enough data accumulates to overcome the prior, revealing the systematic dL→H0 bias

In [ ]:
# --- Figure 7: Why 10ev differs — N-dependence of mu_H ---
import csv as _csv

# Load HyperPE posterior samples
def _load_post(path, params):
    result = {p: [] for p in params}
    with open(path) as f:
        reader = _csv.DictReader(f)
        for row in reader:
            for p in params:
                if p in row: result[p].append(float(row[p]))
    return {p: np.array(v) for p, v in result.items()}

post = {}; post_td = {}
for n in [10, 50, 100]:
    f = f'{RUNDIR}/outdir_hyperpe_run_SEOBNR_{n}/hyperpe_SEOBNR_{n}_full_posterior_extra_statistics.csv'
    if os.path.exists(f): post[n] = _load_post(f, ['mu_H', 'sigma_H'])
    f = f'{RUNDIR}/outdir_hyperpe_eosfit_truth_delta_{n}/eosfit_truth_delta_{n}_full_posterior_extra_statistics.csv'
    if os.path.exists(f): post_td[n] = _load_post(f, ['mu_H', 'sigma_H'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) mu_H posterior: real PE
ax = axes[0,0]
colors = {10: 'C0', 50: 'C2', 100: 'C3'}
for n in [10, 50, 100]:
    if n in post:
        m = np.median(post[n]['mu_H'])
        lo, hi = np.percentile(post[n]['mu_H'], [5, 95])
        ax.hist(post[n]['mu_H'], bins=35, density=True, alpha=0.4, color=colors[n],
                label=f'{n} ev: {m:.1f} [{lo:.0f}, {hi:.0f}]')
ax.axvline(67.66, color='k', lw=2.5, ls='--', label='Truth (67.66)')
ax.axvline(80, color='gray', lw=1.5, ls=':', label='Prior median (80)')
ax.set_xlabel('$\\mu_H$'); ax.set_ylabel('Posterior density')
ax.set_title('(a) $\\mu_H$: real PE', fontweight='bold')
ax.legend(fontsize=7.5)

# (b) mu_H posterior: truth-delta
ax = axes[0,1]
for n in [10, 50, 100]:
    if n in post_td:
        m = np.median(post_td[n]['mu_H'])
        ax.hist(post_td[n]['mu_H'], bins=40, density=True, alpha=0.4, color=colors[n],
                label=f'{n} ev: {m:.3f}')
ax.axvline(67.66, color='k', lw=2.5, ls='--')
ax.set_xlabel('$\\mu_H$'); ax.set_ylabel('Posterior density')
ax.set_title('(b) $\\mu_H$: truth-delta', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(67.55, 67.80)

# (c) 90% CI width vs N
ax = axes[1,0]
n_arr = np.array([10, 50, 100])
w_real = [np.percentile(post[n]['mu_H'],95)-np.percentile(post[n]['mu_H'],5) if n in post else np.nan for n in n_arr]
w_td = [np.percentile(post_td[n]['mu_H'],95)-np.percentile(post_td[n]['mu_H'],5) if n in post_td else np.nan for n in n_arr]
ax.plot(n_arr, w_real, 's-', color='C3', lw=2.5, ms=10, label='Real PE')
ax.plot(n_arr, w_td, 'o-', color='C0', lw=2.5, ms=10, label='Truth-delta')
ax.plot(n_arr, w_real[0]*np.sqrt(10)/np.sqrt(n_arr), 'k:', lw=1.5, alpha=0.5, label=r'$\propto 1/\sqrt{N}$')
ax.set_xlabel('Number of events'); ax.set_ylabel('90% CI width')
ax.set_title('(c) Posterior width vs N', fontweight='bold')
ax.legend(fontsize=8)

# (d) Per-event H0 medians: all subsets biased
ax = axes[1,1]
colors_ev = ['C0']*10 + ['C2']*40 + ['C3']*50
ax.bar(range(1,101), H0_med_all, color=colors_ev, alpha=0.6, width=1.0, edgecolor='none')
ax.axhline(67.66, color='k', lw=2, ls='--')
for grp, c, lbl in [(slice(0,10), 'C0', 'First 10'), (slice(10,50), 'C2', '11-50'), (slice(50,100), 'C3', '51-100')]:
    ax.axhline(np.median(H0_med_all[grp]), color=c, lw=2, ls='-', label=f'{lbl}: med={np.median(H0_med_all[grp]):.1f}')
ax.axvline(10.5, color='gray', lw=1, ls=':', alpha=0.5)
ax.axvline(50.5, color='gray', lw=1, ls=':', alpha=0.5)
ax.set_xlabel('Event index'); ax.set_ylabel('$H_0$ PE median')
ax.set_title('(d) Per-event $H_0$: ALL subsets biased low', fontweight='bold')
ax.legend(fontsize=7.5, loc='lower left'); ax.set_ylim(0, 160)

fig.suptitle('Why 10-Event H$_0$ Looks Different: Prior-Dominated, Not Correct', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{RUNDIR}/fig_N_dependence.png', dpi=150)
plt.show()

# Print summary
for n in [10, 50, 100]:
    if n in post:
        m = np.median(post[n]['mu_H'])
        lo, hi = np.percentile(post[n]['mu_H'], [5, 95])
        print(f"{n:3d}ev: mu_H={m:.2f}, 90%CI=[{lo:.2f},{hi:.2f}], width={hi-lo:.1f}, truth_in={lo<=67.66<=hi}")